# Israeli Walk + Transit — API usage notebook

This notebook talks **only** to the local backend API (`API_BASE_URL`).
It never reads `.env`, PostgreSQL, or the HeiGIT key directly.

Prerequisites:
1. Database migrated and GTFS imported (`npm run db:migrate && npm run db:import:fixture` or full import)
2. API running (`npm run dev`)
3. Notebook deps installed (`pip install -r notebooks/requirements.txt`)

In [9]:
from __future__ import annotations

import json
from copy import deepcopy

import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipyleaflet import Map, Marker, GeoJSON, CircleMarker, LayerGroup, basemaps
from ipywidgets import Layout

from api_client import TransitApiClient

API_BASE_URL = "http://localhost:3001"
client = TransitApiClient(API_BASE_URL)
print("API_BASE_URL =", API_BASE_URL)

API_BASE_URL = http://localhost:3001


## 1. Health, config, GTFS status

In [10]:
health = client.health()
config = client.config()
gtfs = client.gtfs_status()
display({"health": health, "config": config, "gtfs": gtfs})

{'health': {'status': 'ok',
  'database': True,
  'timestamp': '2026-08-08T18:58:46.766Z'},
 'config': {'israelBounds': {'minLng': 34.2,
   'minLat': 29.4,
   'maxLng': 35.95,
   'maxLat': 33.35},
  'defaultWalkingSeconds': 900,
  'maxWalkingSeconds': 1800,
  'endpointRadiusMeters': 500,
  'allowedRouteTypes': [3],
  'feedImportedAt': '2026-08-08T15:59:26.764Z'},
 'gtfs': {'active': {'id': '5ae7517e-fbea-4774-b721-88bce43a5eec',
   'sourceUrl': 'https://gtfs.mot.gov.il/gtfsfiles/israel-public-transportation.zip',
   'sourceSha256': '578c87dafbe416a6bfb828260d0e32ea7f0661e05b99786fd06e14ceb51c0be9',
   'importedAt': '2026-08-08T15:59:26.764Z',
   'stopCount': 35233,
   'routeCount': 7190,
   'tripCount': 308103,
   'stopTimeCount': 11351157,
   'validationNotes': 'Imported agencies=36 stops=35233 routes=7190 trips=308103 stop_times=11351157'},
  'hasActiveFeed': True}}

## 2. Place search helper

In [11]:
def search_and_pick(query: str, limit: int = 5):
    data = client.search_places(query, limit=limit)
    rows = []
    for item in data.get("results", []):
        loc = item["location"]
        rows.append({
            "label": item["label"],
            "lng": loc["lng"],
            "lat": loc["lat"],
            "confidence": item.get("confidence"),
            "id": item["id"],
        })
    return pd.DataFrame(rows)

# Example search — Hebrew addresses may vary by Pelias coverage
example = search_and_pick("Dizengoff Tel Aviv")
display(example)

,label,lng,lat,confidence,id
0,"דיזנגוף, תל־אביב–יפו, הצפון הישן - החלק הצפוני...",34.775868,32.090815,0.402666,nominatim:209827237
1,"דיזנגוף, תל־אביב–יפו, הצפון הישן - החלק הדרומי...",34.773840,32.082030,0.402666,nominatim:210057808
2,"דיזנגוף, תל־אביב–יפו, הצפון הישן - החלק הדרומי...",34.775182,32.075354,0.402666,nominatim:209806593
3,"דיזנגוף, תל־אביב–יפו, הצפון הישן - החלק הצפוני...",34.774588,32.085673,0.402666,nominatim:210193452
4,"דיזנגוף, תל־אביב–יפו, הצפון הישן - החלק הדרומי...",34.775266,32.075371,0.402666,nominatim:209806853


## 3. Planner controls

The walking slider only updates local UI state. Click **Run plan** to call the API.

In [12]:
# Default coordinates (Tel Aviv area). Replace via search or manual lat/lng.
origin = {"lng": 34.7818, "lat": 32.0853}
destination = {"lng": 34.7800, "lat": 32.0750}

origin_query = widgets.Text(value="Dizengoff Center, Tel Aviv", description="Origin q", layout=Layout(width="70%"))
dest_query = widgets.Text(value="Rothschild Blvd, Tel Aviv", description="Dest q", layout=Layout(width="70%"))
origin_select = widgets.Dropdown(options=[], description="Origin")
dest_select = widgets.Dropdown(options=[], description="Dest")
mode = widgets.Dropdown(options=[("Walk + Transit", "walk_transit"), ("Transit + Walk", "transit_walk")], value="walk_transit", description="Mode")
walking = widgets.IntSlider(value=15, min=5, max=30, step=1, description="Walk min")
origin_lng = widgets.FloatText(value=origin["lng"], description="Orig lng")
origin_lat = widgets.FloatText(value=origin["lat"], description="Orig lat")
dest_lng = widgets.FloatText(value=destination["lng"], description="Dest lng")
dest_lat = widgets.FloatText(value=destination["lat"], description="Dest lat")
run_btn = widgets.Button(description="Run plan", button_style="primary")
search_origin_btn = widgets.Button(description="Search origin")
search_dest_btn = widgets.Button(description="Search dest")
status_out = widgets.Output()
map_out = widgets.Output()
table_out = widgets.Output()

last_response = {"raw": None}
scenario_rows = []

m = Map(center=(32.08, 34.78), zoom=13, basemap=basemaps.OpenStreetMap.Mapnik, layout=Layout(height="520px"))
overlay = LayerGroup()
m.add(overlay)


def _options_from_df(df: pd.DataFrame):
    opts = []
    for _, row in df.iterrows():
        label = f"{row['label']} ({row['lat']:.5f},{row['lng']:.5f})"
        opts.append((label, {"lng": float(row["lng"]), "lat": float(row["lat"]), "label": row["label"]}))
    return opts


def on_search_origin(_):
    df = search_and_pick(origin_query.value)
    origin_select.options = _options_from_df(df)
    if origin_select.options:
        origin_select.index = 0


def on_search_dest(_):
    df = search_and_pick(dest_query.value)
    dest_select.options = _options_from_df(df)
    if dest_select.options:
        dest_select.index = 0


def on_origin_selected(change):
    if change["name"] == "value" and change["new"]:
        origin_lng.value = change["new"]["lng"]
        origin_lat.value = change["new"]["lat"]


def on_dest_selected(change):
    if change["name"] == "value" and change["new"]:
        dest_lng.value = change["new"]["lng"]
        dest_lat.value = change["new"]["lat"]


def render_map(payload: dict):
    overlay.clear_layers()
    o = {"lng": origin_lng.value, "lat": origin_lat.value}
    d = {"lng": dest_lng.value, "lat": dest_lat.value}
    overlay.add(Marker(location=(o["lat"], o["lng"]), title="Origin"))
    overlay.add(Marker(location=(d["lat"], d["lng"]), title="Destination"))
    iso = payload.get("isochrone")
    if iso:
        overlay.add(GeoJSON(data=iso, style={"color": "#1d4ed8", "fillOpacity": 0.15, "weight": 2}))
        # Fit roughly to first ring if present
        try:
            coords = iso["features"][0]["geometry"]["coordinates"][0]
            lats = [c[1] for c in coords]
            lngs = [c[0] for c in coords]
            m.fit_bounds([[min(lats), min(lngs)], [max(lats), max(lngs)]])
        except Exception:
            m.center = (o["lat"], o["lng"])
    for stop in payload.get("validStops", []):
        color = {"boarding": "#16a34a", "alighting": "#dc2626", "both": "#9333ea"}.get(stop["role"], "#334155")
        overlay.add(CircleMarker(location=(stop["lat"], stop["lng"]), radius=7, color=color, fill_color=color, fill_opacity=0.9))


def routes_frame(payload: dict) -> pd.DataFrame:
    rows = []
    for r in payload.get("routes", []):
        rows.append({
            "route": r.get("routeShortName") or r.get("routeId"),
            "long_name": r.get("routeLongName"),
            "headsign": r.get("tripHeadsign"),
            "board": r.get("boardStopName"),
            "alight": r.get("alightStopName"),
            "trip_id": r.get("tripId"),
            "board_stop_id": r.get("boardStopId"),
            "alight_stop_id": r.get("alightStopId"),
        })
    return pd.DataFrame(rows)


def on_run(_):
    with status_out:
        clear_output()
        print("Running...")
    try:
        payload, elapsed = client.plan_direct(
            mode=mode.value,
            origin={"lng": float(origin_lng.value), "lat": float(origin_lat.value)},
            destination={"lng": float(dest_lng.value), "lat": float(dest_lat.value)},
            max_walking_seconds=int(walking.value) * 60,
        )
        last_response["raw"] = payload
        with status_out:
            clear_output()
            print(f"requestId={payload.get('requestId')}")
            print(f"client_elapsed_s={elapsed:.2f} server_elapsed_ms={payload.get('meta', {}).get('elapsedMs')}")
            print(f"routes={payload.get('meta', {}).get('routeCount')} stops={payload.get('meta', {}).get('validStopCount')}")
            print(f"isochroneCached={payload.get('meta', {}).get('isochroneCached')}")
            print("warnings:", payload.get("warnings"))
        with map_out:
            clear_output()
            render_map(payload)
            display(m)
        with table_out:
            clear_output()
            display(routes_frame(payload))
    except Exception as exc:
        with status_out:
            clear_output()
            print("ERROR:", exc)

search_origin_btn.on_click(on_search_origin)
search_dest_btn.on_click(on_search_dest)
origin_select.observe(on_origin_selected)
dest_select.observe(on_dest_selected)
run_btn.on_click(on_run)

display(widgets.VBox([
    widgets.HBox([origin_query, search_origin_btn]),
    origin_select,
    widgets.HBox([dest_query, search_dest_btn]),
    dest_select,
    widgets.HBox([origin_lng, origin_lat, dest_lng, dest_lat]),
    widgets.HBox([mode, walking, run_btn]),
    status_out,
    map_out,
    table_out,
]))

## 4. Raw response inspection

In [7]:
if last_response["raw"] is None:
    print("Run a plan first.")
else:
    print(json.dumps(last_response["raw"], ensure_ascii=False, indent=2)[:4000])

{
  "requestId": "3c60c67b-d740-4132-93b4-2b4c92188de8",
  "feedVersion": {
    "id": "5ae7517e-fbea-4774-b721-88bce43a5eec",
    "importedAt": "2026-08-08T15:59:26.764Z",
    "sourceSha256": "578c87dafbe416a6bfb828260d0e32ea7f0661e05b99786fd06e14ceb51c0be9"
  },
  "mode": "walk_transit",
  "isochrone": {
    "type": "FeatureCollection",
    "features": [
      {
        "type": "Feature",
        "properties": {
          "approximated": true,
          "rangeSeconds": 900,
          "note": "Circular fallback used because ORS isochrone was unavailable"
        },
        "geometry": {
          "type": "Polygon",
          "coordinates": [
            [
              [
                34.790976039136936,
                32.094123
              ],
              [
                34.79087398455344,
                32.09544209779238
              ],
              [
                34.79056956698518,
                32.09673862545581
              ],
              [
                34.79

## 5. Scenario matrix (Tel Aviv / Jerusalem / Haifa)

Uses fixed coordinates so results are comparable even if geocoding quality varies.

In [8]:
scenarios = [
    {"city": "Tel Aviv", "mode": "walk_transit", "walk_min": 10, "origin": {"lng": 34.7818, "lat": 32.0853}, "destination": {"lng": 34.7700, "lat": 32.0650}},
    {"city": "Tel Aviv", "mode": "transit_walk", "walk_min": 15, "origin": {"lng": 34.7818, "lat": 32.0853}, "destination": {"lng": 34.7700, "lat": 32.0650}},
    {"city": "Jerusalem", "mode": "walk_transit", "walk_min": 15, "origin": {"lng": 35.2137, "lat": 31.7683}, "destination": {"lng": 35.2250, "lat": 31.7800}},
    {"city": "Haifa", "mode": "walk_transit", "walk_min": 15, "origin": {"lng": 34.9896, "lat": 32.7940}, "destination": {"lng": 35.0000, "lat": 32.8100}},
]

rows = []
for sc in scenarios:
    try:
        payload, elapsed = client.plan_direct(
            mode=sc["mode"],
            origin=sc["origin"],
            destination=sc["destination"],
            max_walking_seconds=sc["walk_min"] * 60,
        )
        rows.append({
            "city": sc["city"],
            "mode": sc["mode"],
            "walk_min": sc["walk_min"],
            "routes": payload["meta"]["routeCount"],
            "stops": payload["meta"]["validStopCount"],
            "cached": payload["meta"]["isochroneCached"],
            "server_ms": payload["meta"]["elapsedMs"],
            "client_s": round(elapsed, 2),
            "warnings": "; ".join(payload.get("warnings") or []),
            "notes": "",
        })
    except Exception as exc:
        rows.append({
            "city": sc["city"],
            "mode": sc["mode"],
            "walk_min": sc["walk_min"],
            "routes": None,
            "stops": None,
            "cached": None,
            "server_ms": None,
            "client_s": None,
            "warnings": str(exc),
            "notes": "failed",
        })

scenario_df = pd.DataFrame(rows)
display(scenario_df)
scenario_df.to_csv("scenario_results.csv", index=False)
print("Wrote notebooks/scenario_results.csv")

,city,mode,walk_min,routes,stops,cached,server_ms,client_s,warnings,notes
0,Tel Aviv,walk_transit,10,139,27,False,3538,3.55,ORS isochrone unavailable; used circular walki...,
1,Tel Aviv,transit_walk,15,200,32,True,2999,3.01,ORS isochrone unavailable; used circular walki...,
2,Jerusalem,walk_transit,15,200,38,False,6017,6.03,ORS isochrone unavailable; used circular walki...,
3,Haifa,walk_transit,15,200,38,False,5176,5.18,ORS isochrone unavailable; used circular walki...,


Wrote notebooks/scenario_results.csv


## Notebook validation gate checklist

- [ ] Both modes return directionally correct same-trip options
- [ ] Isochrone polygon and stop pins look correct on the map
- [ ] Result counts/latency acceptable on the active feed
- [ ] Route fields are understandable without DB access
- [ ] Changing only the fixed endpoint reuses `isochroneCached=true`